# 12 · Production Concerns: Failures, Limits, Cost and Testing

> **Kernel:** `Phase 5 · LangGraph` · **Mostly offline**: failures are simulated
> so they happen on cue. §5–6 use Ollama.

A graph that works in a notebook meets a harsher world in production: APIs
time out, providers rate-limit, models return garbage, costs creep up, and
someone changes a prompt and breaks a route. This notebook covers LangGraph's
tools for each of those, plus how to **test** a graph whose nodes call a
non-deterministic model.

**You will learn**
- **Retries** per node with `RetryPolicy`, and what to retry (and what not)
- **Timeouts** per node, and why they need async nodes
- **Error handlers**: recovering to a fallback path instead of crashing
- **Model fallbacks**: switching provider when one fails
- **Budgets**: tracking tokens per node and stopping gracefully
- **Caching** deterministic nodes, and **runtime context** for per-run settings
- **Testing**: unit tests with fake models, and route tests for the graph
- A production checklist

**Prerequisites:** notebooks 02–08.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import asyncio, time, operator
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy, CachePolicy, Command
from langgraph.errors import NodeError, NodeTimeoutError

## 1 · Why: what goes wrong in production

| Failure | Example | LangGraph tool |
|---|---|---|
| Transient errors | network blip, 503, rate limit | `RetryPolicy` on the node |
| Hanging calls | a tool that never returns | `timeout=` on the node |
| Persistent failure of one step | a provider outage | `error_handler` → a fallback path |
| A model provider down | Ollama restarting | `model.with_fallbacks([...])` |
| Runaway cost | a loop that keeps calling the model | budgets in state, call limits |
| Silent regressions | a prompt edit breaks routing | tests with fake models |

## 2 · Retries: `RetryPolicy`

A node can be given a retry policy: how many attempts, how long to wait between
them (**exponential backoff** with jitter), and **which exceptions** to retry.
The node below fails twice with a connection error, then succeeds:

In [ ]:
class Job(TypedDict):
    result: str
    status: str

calls = {'fetch': 0}

def flaky_fetch(state: Job) -> dict:
    calls['fetch'] += 1
    if calls['fetch'] < 3:
        raise ConnectionError(f"attempt {calls['fetch']}: upstream unavailable")
    return {'result': 'data'}


b = StateGraph(Job)
b.add_node('fetch', flaky_fetch,
           retry_policy=RetryPolicy(max_attempts=4,          # 1 try + up to 3 retries
                                    initial_interval=0.2,    # first wait (seconds)
                                    backoff_factor=2.0,      # 0.2s, 0.4s, 0.8s, …
                                    retry_on=ConnectionError))  # ONLY these are retried
b.add_edge(START, 'fetch')
g = b.compile()

t0 = time.perf_counter()
print(g.invoke({'result': '', 'status': ''}), f"after {calls['fetch']} attempts, {time.perf_counter() - t0:.1f}s")

**What to retry.** Retry **transient** failures (connection errors, timeouts,
rate limits, 5xx responses). Don't retry **deterministic** ones (a `KeyError`,
invalid input, a 400 response), because they'll fail the same way every time and
you just add delay. That's why `retry_on` matters. The default `retry_on`
already skips most programming errors (`ValueError`, `TypeError`, …), but being
explicit documents your intent.

**Retries and side effects.** A retried node runs again from the top. If it
sent an email before failing, the email goes out again. Keep side effects
idempotent (notebook 08).

## 3 · Timeouts

A call that hangs is worse than one that fails, because nothing happens at all.
Give a node `timeout=` seconds. If it's exceeded, LangGraph raises
`NodeTimeoutError`, which a retry policy can then retry.

Timeouts only work on **async** nodes. Python can't safely interrupt a running
synchronous function, so LangGraph refuses rather than pretending:

In [ ]:
async def slow_tool(state: Job) -> dict:
    await asyncio.sleep(3)                      # pretend: a service that hangs
    return {'result': 'too late'}

b = StateGraph(Job)
b.add_node('slow_tool', slow_tool, timeout=0.5)
b.add_edge(START, 'slow_tool')
try:
    await b.compile().ainvoke({'result': '', 'status': ''})
except NodeTimeoutError as err:
    print('async node :', err)

b = StateGraph(Job)
b.add_node('sync_node', lambda s: {'result': 'x'}, timeout=0.5)
b.add_edge(START, 'sync_node')
try:
    b.compile().invoke({'result': '', 'status': ''})
except ValueError as err:
    print('sync node  :', str(err).split(' because')[0], '(sync nodes cannot be cancelled)')

For sync code, set timeouts **inside** the node on the client you call (most
HTTP and model clients accept a `timeout=` argument).

## 4 · Error handlers: a fallback path instead of a crash

Retries handle blips. When a step fails for good, you often still want the run
to **finish**: with cached data, a partial answer, or a ticket for a human.
`add_node(..., error_handler=fn)` runs `fn` after the node's retries are
exhausted. The handler receives the state and a `NodeError` (the failed node's
name and the exception), and can update state and **route** with `Command`:

```
 START → fetch ──ok──► summarise → END
           │ (retries exhausted)
           └──► error_handler ──Command(goto)──► use_cache → END
```

In [ ]:
calls['fetch'] = 0

def always_down(state: Job) -> dict:
    calls['fetch'] += 1
    raise ConnectionError(f"attempt {calls['fetch']}: provider outage")

def on_fetch_error(state: Job, error: NodeError) -> Command[Literal['use_cache']]:
    return Command(update={'status': f'{error.node} failed ({error.error}); served cached data'},
                   goto='use_cache')

def summarise(state: Job) -> dict: return {'result': 'fresh summary'}
def use_cache(state: Job) -> dict: return {'result': 'summary from yesterday (cached)'}


b = StateGraph(Job)
b.add_node('fetch', always_down,
           retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.1, retry_on=ConnectionError),
           error_handler=on_fetch_error)
b.add_node('summarise', summarise)
b.add_node('use_cache', use_cache)
b.add_edge(START, 'fetch')
b.add_edge('fetch', 'summarise')
b.add_edge('use_cache', END)
robust = b.compile()

out = robust.invoke({'result': '', 'status': ''})
print(out['result'])
print(out['status'])

Three attempts (the retry policy), then the handler sent the run down the
fallback path. The caller got a useful, honestly labelled answer instead of a
stack trace.

## 5 · Model fallbacks

The model itself can fail: the local server is restarting, a cloud provider has
an outage, or you hit a rate limit. Every LangChain chat model has
`.with_fallbacks([...])`: if the primary raises, the next one is tried. Here the
primary deliberately points at a model that doesn't exist:

In [ ]:
from langchain_ollama import ChatOllama
from common import get_model

primary = ChatOllama(model='this-model-does-not-exist', temperature=0)
backup = get_model()
resilient = primary.with_fallbacks([backup])

reply = resilient.invoke('Say "fallback worked" and nothing else.')
print(reply.content, '| answered by:', reply.response_metadata.get('model'))

In production the chain is usually *cloud primary → other provider → local
model*, or *strong model → cheaper model*. Retries on the model
(`model.with_retry(stop_after_attempt=3)`) and fallbacks combine well: retry
blips first, then fall back.

## 6 · Budgets: counting tokens per node and stopping gracefully

Loops and agents can quietly burn tokens. Make cost **visible** (tokens per node
in state) and **bounded** (a router that stops when the budget is spent). Every
`AIMessage` reports its `usage_metadata`, so a node can record what it used:

In [ ]:
def merge_usage(old: dict | None, new: dict) -> dict:
    out = dict(old or {})
    for node, tokens in new.items():
        out[node] = out.get(node, 0) + tokens
    return out


class Refine(TypedDict):
    text: str
    rounds: int
    usage: Annotated[dict, merge_usage]          # tokens used, per node
    status: str

BUDGET = 250                                      # total tokens this run may use
model = get_model()


def improve(state: Refine) -> dict:
    reply = model.invoke(f"Rewrite to be clearer, max 40 words:\n{state['text']}")
    return {'text': reply.content, 'rounds': state['rounds'] + 1,
            'usage': {'improve': reply.usage_metadata['total_tokens']}}


def budget_check(state: Refine) -> Literal['improve', 'stop']:
    spent = sum(state['usage'].values())
    print(f"  round {state['rounds']}: {spent} tokens spent")
    return 'stop' if spent >= BUDGET or state['rounds'] >= 10 else 'improve'


def stop(state: Refine) -> dict:
    return {'status': f"stopped after {state['rounds']} rounds, {sum(state['usage'].values())} tokens"}


b = StateGraph(Refine)
b.add_node('improve', improve)
b.add_node('stop', stop)
b.add_edge(START, 'improve')
b.add_conditional_edges('improve', budget_check, ['improve', 'stop'])
b.add_edge('stop', END)
refiner = b.compile()

out = refiner.invoke({'text': 'Our robots, which are delivery robots, deliver things that people order, '
                              'like groceries and other things, to their houses.',
                      'rounds': 0, 'usage': {}})
print(out['status'], '| per node:', out['usage'])

Multiply tokens by your provider's price for a cost meter. For `create_agent`,
`ModelCallLimitMiddleware` and `ToolCallLimitMiddleware` (notebook 04) do the
same job with less code.

## 7 · Caching deterministic nodes

If a node is **deterministic** and expensive (embedding a document, fetching a
reference file), there's no reason to run it twice for the same input. Give it
a `CachePolicy` and compile with a cache:

In [ ]:
from langgraph.cache.memory import InMemoryCache

class Doc(TypedDict):
    doc_id: str
    summary: str

runs = {'n': 0}
def expensive_summary(state: Doc) -> dict:
    runs['n'] += 1
    time.sleep(0.5)
    return {'summary': f"summary of {state['doc_id']}"}

b = StateGraph(Doc)
b.add_node('expensive_summary', expensive_summary, cache_policy=CachePolicy(ttl=600))   # 10 min
b.add_edge(START, 'expensive_summary')
cached = b.compile(cache=InMemoryCache())

for doc in ['A', 'A', 'B', 'A']:
    t0 = time.perf_counter()
    cached.invoke({'doc_id': doc, 'summary': ''})
    print(f'doc {doc}: {time.perf_counter() - t0:.2f}s')
print('node actually ran', runs['n'], 'times')

The cache key is the node's **input**, so the same `doc_id` hits the cache.
Only cache nodes whose output depends *only* on their input: never cache a node
with side effects, or one whose answer should change over time (beyond the TTL).

## 8 · Runtime context: per-run settings that aren't state

Some values belong to a *run*, not to the state: the user's id, which model to
use, a feature flag, a database handle. Putting them in state would checkpoint
them, which is pointless and sometimes impossible. Declare a **context schema**
and pass `context=` at invoke time. Nodes read it from `runtime.context`:

In [ ]:
from dataclasses import dataclass
from langgraph.runtime import Runtime

@dataclass
class RunContext:
    user_id: str
    tier: Literal['free', 'pro']

class Answer(TypedDict):
    question: str
    answer: str

def respond(state: Answer, runtime: Runtime[RunContext]) -> dict:
    ctx = runtime.context
    depth = 'a detailed' if ctx.tier == 'pro' else 'a one-line'
    return {'answer': f"[{ctx.user_id}] {depth} answer to: {state['question']}"}

b = StateGraph(Answer, context_schema=RunContext)
b.add_node('respond', respond)
b.add_edge(START, 'respond')
app = b.compile()

print(app.invoke({'question': 'Q3 status?'}, context=RunContext(user_id='ada', tier='pro'))['answer'])
print(app.invoke({'question': 'Q3 status?'}, context=RunContext(user_id='bo', tier='free'))['answer'])

## 9 · Testing graphs whose nodes call models

LLM output varies, so don't test *the model*. Test **your logic around it**:
does the router route correctly for each possible decision? Does the loop stop?
Does the state have the right keys? The trick is a **fake model** that returns
scripted responses. LangChain ships several; `FakeMessagesListChatModel`
returns a fixed list of messages in order.

The graph under test is built by a **factory function that takes the model as
a parameter**. That's the key design choice: production passes a real model,
and tests pass a fake one.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage


class Ticket(TypedDict):
    text: str
    category: str
    reply: str


def build_triage(model):                          # the model is a PARAMETER
    def classify(state: Ticket) -> dict:
        word = model.invoke(f"One word, billing or technical: {state['text']}").content.strip().lower()
        return {'category': word if word in ('billing', 'technical') else 'human'}   # safe default

    def billing(state):   return {'reply': 'refund desk'}
    def technical(state): return {'reply': 'support engineer'}
    def human(state):     return {'reply': 'human queue'}

    b = StateGraph(Ticket)
    for fn in (classify, billing, technical, human):
        b.add_node(fn.__name__, fn)
    b.add_edge(START, 'classify')
    b.add_conditional_edges('classify', lambda s: s['category'], ['billing', 'technical', 'human'])
    return b.compile()


def fake(reply: str):
    return FakeMessagesListChatModel(responses=[AIMessage(reply)])


# ---- tests: each one scripts the model's answer and checks the ROUTE -------
def test_billing_route():
    assert build_triage(fake('billing')).invoke({'text': 'x'})['reply'] == 'refund desk'

def test_technical_route():
    assert build_triage(fake('Technical')).invoke({'text': 'x'})['reply'] == 'support engineer'

def test_unexpected_answer_goes_to_human():
    # the model rambles instead of answering with one word: must not crash, must go to a human
    assert build_triage(fake('Hmm, hard to say, maybe billing?')).invoke({'text': 'x'})['reply'] == 'human queue'


for test in (test_billing_route, test_technical_route, test_unexpected_answer_goes_to_human):
    test()
    print('✅', test.__name__)

These tests run in milliseconds, cost nothing, and never flake. Put them in a
`tests/` folder and run them with `pytest` on every change. Three layers of
testing work well together:

| Layer | What it checks | Uses |
|---|---|---|
| **Node unit tests** | one node: state in → update out | plain function calls |
| **Route tests** | the graph's logic for every model decision | fake models (as above) |
| **Evaluations** | the *quality* of real model output on a fixed dataset | the real model, scored |

Evaluations measure answer quality over a dataset, and they're a discipline of
their own.

## 10 · Deploying a graph

A compiled graph is just a Python object, so you can serve it however you like:

- **Your own API**: wrap `ainvoke`/`astream` in a FastAPI endpoint, use a
  `PostgresSaver` for checkpoints, and pass `thread_id` from the client. Stream
  tokens to the browser with server-sent events.
- **A job worker**: run graphs from a queue; durable checkpoints let a crashed
  worker's job be resumed by another.
- **LangGraph Platform / Server**: LangChain's hosted and self-hostable runtime
  for graphs. It provides the API, task queue, persistence, streaming and Studio
  integration. It's useful when you'd rather not build that yourself; it's a
  product with its own pricing, not a requirement.

## 11 · Production checklist

- [ ] **Durable checkpointer** (SQLite for one machine, Postgres for many), with `durability='sync'` where re-running is costly
- [ ] **Retry policies** on nodes that call networks, and only for transient errors
- [ ] **Timeouts** on async nodes and on client calls
- [ ] **Error handlers / fallback paths** for steps whose failure shouldn't kill the run
- [ ] **Model fallbacks** across providers
- [ ] **Loop guards**: counters in state, `recursion_limit` as the last resort
- [ ] **Budgets**: tokens tracked per node, graceful stop
- [ ] **`max_concurrency`** on fan-outs to respect rate limits
- [ ] **Idempotent side effects** (at-least-once execution)
- [ ] **Human approval** for irreversible tools
- [ ] **Least-privilege tools** per agent
- [ ] **Tracing** in production, with a policy on what data may leave your systems
- [ ] **Tests**: node unit tests and route tests with fake models, run in CI
- [ ] **Thread retention**: delete or archive old threads

## 12 · Check yourself

<details><summary><b>1.</b> Which of these should a retry policy retry: a <code>ConnectionError</code>, a <code>KeyError</code> in your code, a 429 rate-limit response? Why?</summary>

The connection error and the 429 are transient, and retrying (with backoff) can succeed. The `KeyError` is a deterministic bug: it fails the same way every time, so retrying only adds delay and hides the problem.
</details>

<details><summary><b>2.</b> Why does LangGraph refuse a <code>timeout</code> on a sync node, and what do you do instead?</summary>

Python can't safely cancel a running synchronous function, so the timeout couldn't actually stop it. Make the node async, or set a timeout on the client it calls (HTTP/model client `timeout=`).
</details>

<details><summary><b>3.</b> Retry policy vs error handler: what does each do, and in what order?</summary>

The retry policy re-runs the node on transient failures. The error handler runs after the retries are exhausted, and can update state and route to a fallback path (via `Command`) so the run finishes instead of crashing.
</details>

<details><summary><b>4.</b> Why pass the model into a graph-building function instead of creating it inside?</summary>

So tests can pass a fake model with scripted responses. That lets you test routing and control flow deterministically, instantly and for free, and production passes the real model.
</details>

<details><summary><b>5.</b> Name something that belongs in runtime context rather than state, and why.</summary>

For example a user id, a subscription tier, a model choice or a database handle. These are per-run settings or resources, not part of the work's evolving data. Keeping them out of state avoids checkpointing them, and handles couldn't be serialised anyway.
</details>

<details><summary><b>6.</b> Which nodes are safe to cache?</summary>

Deterministic nodes without side effects, whose output depends only on their input (within the TTL), such as summarising a fixed document or computing an embedding. Never cache nodes that act on the world, or whose answer must reflect changing data.
</details>

## Takeaway

Production graphs fail gracefully: **retry** transient errors, **time out**
hanging calls, **handle** persistent failures with a fallback path, **fall back**
across models, and **budget** tokens with a graceful stop. Cache deterministic
work, pass per-run settings as **context**, and build graphs from factories that
accept a model, so routes can be tested with fake models.

That completes the teaching notebooks. Next → `project.ipynb`: the
research-and-report assistant, using everything from 01–12 on the Nimbus
Robotics knowledge base.